# Introduction to Data Science 2026

# Week 4


In this week's exercise, we look at prompting and zero- and few-shot task settings. Below is a text generation example from https://github.com/TurkuNLP/intro-to-nlp/blob/master/text_generation_pipeline_example.ipynb demonstrating how to load a text generation pipeline with a pre-trained model and generate text with a given prompt. Your task is to load a similar pre-trained generative model and assess whether the model succeeds at a set of tasks in zero-shot, one-shot, and two-shot settings.

**Note: Downloading and running the pre-trained model locally may take some time. Alternatively, you can open and run this notebook on [Google Colab](https://colab.research.google.com/), as assumed in the following example.**


## Text generation example

This is a brief example of how to run text generation with a causal language model and `pipeline`.

Install [transformers](https://huggingface.co/docs/transformers/index) python package. This will be used to load the model and tokenizer and to run generation.


In [ ]:
!pip install --quiet transformers

Import the `AutoTokenizer`, `AutoModelForCausalLM`, and `pipeline` classes. The first two support loading tokenizers and generative models from the [Hugging Face repository](https://huggingface.co/models), and the last wraps a tokenizer and a model for convenience.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

Load a generative model and its tokenizer. You can substitute any other generative model name here (e.g. [other TurkuNLP GPT-3 models](https://huggingface.co/models?sort=downloads&search=turkunlp%2Fgpt3)), but note that Colab may have issues running larger models.


In [ ]:
MODEL_NAME = 'TurkuNLP/gpt3-finnish-large'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

Instantiate a text generation pipeline using the tokenizer and model.


In [ ]:
pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device=model.device
)

We can now call the pipeline with a text prompt; it will take care of tokenizing, encoding, generation, and decoding:


In [ ]:
output = pipe('Terve, miten menee?', max_new_tokens=25)

print(output)

Just print the text


In [ ]:
print(output[0]['generated_text'])

We can also call the pipeline with any arguments that the model `generate` function supports. For details on text generation using `transformers`, see e.g. [this tutorial](https://huggingface.co/blog/how-to-generate).

Example with sampling and a high `temperature` parameter to generate more chaotic output:


In [ ]:
output = pipe(
    'Terve, miten menee?',
    do_sample=True,
    temperature=10.0,
    max_new_tokens=25
)

print(output[0]['generated_text'])

## Exercise 1

Your task is to assess whether a generative model succeeds in the following tasks in zero-shot, one-shot, and two-shot settings:

- binary sentiment classification (positive / negative)

- person name recognition

- two-digit addition (e.g. 11 + 22 = 33)

For example, for assessing whether a generative model can name capital cities, we could use the following prompts:

- zero-shot:
  > """\
  > Identify the capital cities of countries.
  >
  > Question: What is the capital of Finland?\
  > Answer:\
  > """
- one-shot:
  > """\
  > Identify the capital cities of countries.
  >
  > Question: What is the capital of Sweden?\
  > Answer: Stockholm
  >
  > Question: What is the capital of Finland?\
  > Answer:\
  > """
- two-shot:
  > """\
  > Identify the capital cities of countries.
  >
  > Question: What is the capital of Sweden?\
  > Answer: Stockholm
  >
  > Question: What is the capital of Denmark?\
  > Answer: Copenhagen
  >
  > Question: What is the capital of Finland?\
  > Answer:\
  > """

You can do the tasks either in English or Finnish and use a generative model of your choice from the Hugging Face models repository, for example the following models:

- English: `gpt2-large`
- Finnish: `TurkuNLP/gpt3-finnish-large`

You can either come up with your own instructions for the tasks or use the following:

- English:
  - binary sentiment classification: "Do the following texts express a positive or negative sentiment?"
  - person name recognition: "List the person names occurring in the following texts."
  - two-digit addition: "This is a first grade math exam."
- Finnish:
  - binary sentiment classification: "Ilmaisevatko seuraavat tekstit positiivista vai negatiivista tunnetta?"
  - person name recognition: "Listaa seuraavissa teksteissä mainitut henkilönnimet."
  - two-digit addition: "Tämä on ensimmäisen luokan matematiikan koe."

Come up with at least two test cases for each of the three tasks, and come up with your own one- and two-shot examples.


In [ ]:
## Binary sentiment classification
print("-------------Binary sentiment classification-------------")
zero_shot_prompt_sentiment_1 = """
Kysymys: Ilmaisevatko seuraavat tekstit positiivista vai negatiivista tunnetta?

"Olen iloinen!" ilmaisee
"""

output = pipe(
    zero_shot_prompt_sentiment_1,
    max_new_tokens=25
)

print("-------------ZERO SHOT PROMPT 1-------------")
print(output[0]["generated_text"])
zero_shot_prompt_sentiment_2 = """
Kysymys: Ilmaisevatko seuraavat tekstit positiivista vai negatiivista tunnetta?

"En olisi halunnut hänen tekevän sitä." ilmaisee
"""

output = pipe(
    zero_shot_prompt_sentiment_2,
    max_new_tokens=25
)

print("-------------ZERO SHOT PROMPT 2-------------")
print(output[0]["generated_text"])

one_shot_prompt_sentiment_1 = """
Kysymys: Ilmaisevatko seuraavat tekstit positiivista vai negatiivista tunnetta?

"Olen iloinen!" ilmaisee positiivista tunnetta.
"Olen surullinen :(" ilmaisee
"""
output = pipe(
    one_shot_prompt_sentiment_1,
    max_new_tokens=25
)
print("-------------ONE SHOT PROMPT 1-------------")
print(output[0]["generated_text"])
one_shot_prompt_sentiment_2 = """
Kysymys: Ilmaisevatko seuraavat tekstit positiivista vai negatiivista tunnetta?

"En olisi halunnut hänen tekevän sitä." ilmaisee negatiivista tunnetta.
"Olen toteuttanut unelmani!" ilmaisee
"""
output = pipe(
    one_shot_prompt_sentiment_2,
    max_new_tokens=25
)
print("-------------ONE SHOT PROMPT 2-------------")
print(output[0]["generated_text"])

two_shot_prompt_sentiment_1 = """
Kysymys: Ilmaisevatko seuraavat tekstit positiivista vai negatiivista tunnetta?

"Olen iloinen!" ilmaisee positiivista tunnetta.
"Olen surullinen :(" ilmaisee negatiivista tunnetta.
"Vihaan sinua." ilmaisee
"""
output = pipe(
    two_shot_prompt_sentiment_1,
    max_new_tokens=25
)
print("-------------TWO SHOT PROMPT 1-------------")
print(output[0]["generated_text"])
two_shot_prompt_sentiment_2 = """
Kysymys: Ilmaisevatko seuraavat tekstit positiivista vai negatiivista tunnetta?

"En olisi halunnut hänen tekevän sitä." ilmaisee negatiivista tunnetta.
"Olen toteuttanut unelmani!" ilmaisee positiivista tunnetta.
"Kiitos." ilmaisee
"""
output = pipe(
    two_shot_prompt_sentiment_2,
    max_new_tokens=25
)
print("-------------TWO SHOT PROMPT 2-------------")
print(output[0]["generated_text"])







## Person name recognition
print("-------------Person name recognition-------------")
zero_shot_prompt_person_1 = """
Listaa seuraavissa teksteissä mainitut henkilönnimet. 

Teksti 1: "Maunu, Liisa ja Matias menivät ostoksille."
Vastaus:
"""
output = pipe(
    zero_shot_prompt_person_1,
    max_new_tokens=25
)
print("-------------ZERO SHOT PROMPT 1-------------")
print(output[0]["generated_text"])
zero_shot_prompt_person_2 = """
Listaa seuraavissa teksteissä mainitut henkilönnimet. 

Teksti 1: "Olin Juhanin ja Pekan kanssa."
Vastaus:
"""
output = pipe(
    zero_shot_prompt_person_2,
    max_new_tokens=25
)

print("-------------ZERO SHOT PROMPT 2-------------")
print(output[0]["generated_text"])

one_shot_prompt_person_1 = """
Listaa seuraavissa teksteissä mainitut henkilönnimet. 

Teksti 1: "Maunu, Liisa ja Matias menivät ostoksille."
Vastaus: Maunu, Liisa ja Matias.

Teksti 2: "Tämä ei haitannut Oskaria."
Vastaus:
"""
output = pipe(
    one_shot_prompt_person_1,
    max_new_tokens=25
)
print("-------------ONE SHOT PROMPT 1-------------")
print(output[0]["generated_text"])
one_shot_prompt_person_2 = """
Listaa seuraavissa teksteissä mainitut henkilönnimet. 

Teksti 1: "Olin Juhanin ja Pekan kanssa."
Vastaus: Juhani ja Pekka.

Teksti 2: "Veera ei päässyt paikalle."
Vastaus: 
"""
output = pipe(
    one_shot_prompt_person_2,
    max_new_tokens=25
)
print("-------------ONE SHOT PROMPT 2-------------")
print(output[0]["generated_text"])

two_shot_prompt_person_1 = """
Listaa seuraavissa teksteissä mainitut henkilönnimet. 

Teksti 1: "Maunu, Liisa ja Matias menivät ostoksille."
Vastaus: Maunu, Liisa ja Matias.

Teksti 2: "Tämä ei haitannut Oskaria."
Vastaus: Oskari.

Teksti 3: "Ja Heidi ei edes tiennyt asiasta."
Vastaus: 
"""
output = pipe(
    two_shot_prompt_person_1,
    max_new_tokens=25
)
print("-------------TWO SHOT PROMPT 1-------------")
print(output[0]["generated_text"])
two_shot_prompt_person_2 = """
Listaa seuraavissa teksteissä mainitut henkilönnimet. 

Teksti 1: "Olin Juhanin ja Pekan kanssa."
Vastaus: Juhani ja Pekka.

Teksti 2: "Veera ei päässyt paikalle."
Vastaus: Veera.

Teksti 3: "Mutta hän tuli ensi viikolla."
Vastaus: 
"""
output = pipe(
    two_shot_prompt_person_2,
    max_new_tokens=25
)
print("-------------TWO SHOT PROMPT 2-------------")
print(output[0]["generated_text"])










## Two-digit addition
print("-------------Two-digit addition-------------")
zero_shot_prompt_math_1 = """
Tämä on ensimmäisen luokan matematiikan koe.

11 + 22 = 
"""
output = pipe(
    zero_shot_prompt_math_1,
    max_new_tokens=15
)
print("-------------ZERO SHOT PROMPT 1-------------")
print(output[0]["generated_text"])
zero_shot_prompt_math_2 = """
Tämä on ensimmäisen luokan matematiikan koe.

10 + 10 = 
"""
output = pipe(
    zero_shot_prompt_math_2,
    max_new_tokens=15
)
print("-------------ZERO SHOT PROMPT 2-------------")
print(output[0]["generated_text"])
one_shot_prompt_math_1 = """
Tämä on ensimmäisen luokan matematiikan koe.

11 + 22 = 33
22 + 44 = 
"""
output = pipe(
    one_shot_prompt_math_1,
    max_new_tokens=15
)
print("-------------ONE SHOT PROMPT 1-------------")
print(output[0]["generated_text"])
one_shot_prompt_math_2 = """
Tämä on ensimmäisen luokan matematiikan koe.

10 + 10 = 20
20 + 20 = 
"""
output = pipe(
    one_shot_prompt_math_2,
    max_new_tokens=15
)
print("-------------ONE SHOT PROMPT 2-------------")
print(output[0]["generated_text"])
two_shot_prompt_math_1 = """
Tämä on ensimmäisen luokan matematiikan koe.

11 + 22 = 33
22 + 44 = 66
33 + 66 = 
"""
output = pipe(
    two_shot_prompt_math_1,
    max_new_tokens=15
)
print("-------------TWO SHOT PROMPT 1-------------")
print(output[0]["generated_text"])
two_shot_prompt_math_2 = """
Tämä on ensimmäisen luokan matematiikan koe.

10 + 10 = 20
20 + 20 = 40
30 + 30 = 
"""
output = pipe(
    two_shot_prompt_math_2,
    max_new_tokens=15
)
print("-------------TWO SHOT PROMPT 2-------------")
print(output[0]["generated_text"])

WRITTEN ANSWER:
The model can generate correct answers for the first two problems and in all of the different n-shot settings, although one-shot and two-shot settings seem to result in better outputs more often than the zero-shot setting. But it also often gets questions wrong, or simply does not answer. Two-digit addition seems to be especially difficult, and I haven't seen a single correct answer to the addition problems.

The model also likes to generate new questions or problems that have the same answers as the questions in the prompt, then answers those correctly. This can be seen especially in the binary sentiment classification part. Overall, I think the model performs best with the binary sentiment classification, decently with name recognition, and it cannot handle two-digit addition at all.

**Submit this exercise by submitting your code and your answers to the above questions as comments on the MOOC platform. You can return this Jupyter notebook (.ipynb) or .py, .R, etc depending on your programming preferences.**
